In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [3]:
df = pd.read_csv("Housing.csv")

df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [4]:
df.isnull().sum()

price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

In [5]:
binary_columns = [
    'mainroad',
    'guestroom',
    'basement',
    'hotwaterheating',
    'airconditioning',
    'prefarea'
]

for col in binary_columns:
    df[col] = df[col].map({'yes':1,'no':0})

df = pd.get_dummies(
    df,
    columns=['furnishingstatus'],
    drop_first=True
)

df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,False,False
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,False,False
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,True,False
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,False,False
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,False,False


In [6]:
X = df.drop("price", axis=1)

y = df["price"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [8]:
model = LinearRegression()

model.fit(X_train, y_train)

print("Model Trained Successfully")

Model Trained Successfully


In [9]:
model = LinearRegression()

model.fit(X_train, y_train)

print("Model Trained Successfully")

Model Trained Successfully


In [10]:
joblib.dump(model, "house_model.pkl")

joblib.dump(X.columns, "columns.pkl")

print("Model Saved Successfully")

Model Saved Successfully


In [11]:
new_house = pd.DataFrame({

    "area":[5000],
    "bedrooms":[3],
    "bathrooms":[2],
    "stories":[2],
    "mainroad":[1],
    "guestroom":[0],
    "basement":[1],
    "hotwaterheating":[0],
    "airconditioning":[1],
    "parking":[2],
    "prefarea":[1],
    "furnishingstatus_semi-furnished":[1],
    "furnishingstatus_unfurnished":[0]

})

price = model.predict(new_house)

print("Predicted Price =", price[0])

Predicted Price = 7176345.6813739


In [1]:
from flask import Flask, render_template, request
import joblib
import pandas as pd

app = Flask(__name__)

# Load trained model
model = joblib.load("house_model.pkl")
columns = joblib.load("columns.pkl")


@app.route("/")
def home():
    return render_template("index.html")


@app.route("/predict", methods=["POST"])
def predict():

    area = float(request.form["area"])
    bedrooms = int(request.form["bedrooms"])
    bathrooms = int(request.form["bathrooms"])
    stories = int(request.form["stories"])
    parking = int(request.form["parking"])

    mainroad = 1 if request.form["mainroad"] == "yes" else 0
    guestroom = 1 if request.form["guestroom"] == "yes" else 0
    basement = 1 if request.form["basement"] == "yes" else 0
    hotwaterheating = 1 if request.form["hotwaterheating"] == "yes" else 0
    airconditioning = 1 if request.form["airconditioning"] == "yes" else 0
    prefarea = 1 if request.form["prefarea"] == "yes" else 0

    furnishing = request.form["furnishing"]

    furnished = 0
    semi = 0
    unfurnished = 0

    if furnishing == "furnished":
        furnished = 1

    elif furnishing == "semi-furnished":
        semi = 1

    else:
        unfurnished = 1

    new_house = pd.DataFrame({

        "area":[area],
        "bedrooms":[bedrooms],
        "bathrooms":[bathrooms],
        "stories":[stories],
        "mainroad":[mainroad],
        "guestroom":[guestroom],
        "basement":[basement],
        "hotwaterheating":[hotwaterheating],
        "airconditioning":[airconditioning],
        "parking":[parking],
        "prefarea":[prefarea],
        "furnishingstatus_semi-furnished":[semi],
        "furnishingstatus_unfurnished":[unfurnished]

    })

    prediction = model.predict(new_house)

    price = round(prediction[0],2)

    return render_template(
        "index.html",
        prediction_text=f"Predicted House Price = Rs. {price:,.2f}"
    )


if __name__ == "__main__":
    app.run(host="127.0.0.1", port=8060, debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:8060
Press CTRL+C to quit
 * Restarting with watchdog (inotify)
Traceback (most recent call last):
  File "/home/zakir/anaconda3/lib/python3.13/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
                     "__main__", mod_spec)
  File "/home/zakir/anaconda3/lib/python3.13/runpy.py", line 88, in _run_code
    exec(code, run_globals)
    ~~~~^^^^^^^^^^^^^^^^^^^
  File "/home/zakir/anaconda3/lib/python3.13/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
    ~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/home/zakir/anaconda3/lib/python3.13/site-packages/traitlets/config/application.py", line 1074, in launch_instance
    app.initialize(argv)
    ~~~~~~~~~~~~~~^^^^^^
  File "/home/zakir/anaconda3/lib/python3.13/site-packages/traitlets/config/application.py", line 118, in inner
    return method(app, *args, **kwargs)
  File "/home/zakir/anaconda3/lib/python3.13/site-packages/ipykernel

SystemExit: 1

/home/zakir/anaconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [13]:
import os

# Create folders
os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)

# Create empty files
open("templates/index.html", "w").close()
open("static/style.css", "w").close()

print("Folders and files created successfully!")

Folders and files created successfully!


In [ ]:
import os

# Create folders
os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)

# Create empty files
files = [
    "app.py",
    "templates/index.html",
    "static/style.css"
]

for file in files:
    if not os.path.exists(file):
        open(file, "w").close()

print("Project structure created successfully!")